# Klasifikasi Bunga: VGG16 + SVM (CNN-SVM Hybrid)
## Mata Kuliah Machine Learning — UAS

**Dataset:** Bunga Melati Jakarta, Melati Jepang, Bintaro, dan Tapak Dara  
**Referensi Utama:** Fei et al. (2023) — *A Lightweight Attention-Based Convolutional Neural Networks for Fresh-Cut Flower Classification*, IEEE Access 11, 17283–17293

---
### Latar Belakang
Mengacu pada Fei et al. (2023) Section I: *"CNN was proposed as a feature extractor and used in combination with other machine learning classification algorithms to synthesize algorithms with more classification advantages, such as VGG, AlexNet, and DenseNet, used as feature extractors and SVM, RF, etc., used as feature classification tools. The highest classification accuracy of these methods is up to 99.8%."*

### Alur Penelitian:
1. Import Library
2. Konfigurasi & Struktur Dataset
3. Load & Preprocessing Dataset
4. Data Augmentation
5. **Metode ML (UTS):** Random Forest
6. **Metode Deep Learning (UAS):** VGG16 Fine-Tuning (End-to-End)
7. **Metode Hybrid (UAS):** VGG16 Feature Extractor + SVM (CNN-SVM)
8. Evaluasi & Perbandingan Semua Model
9. Visualisasi Hasil
10. Demo Prediksi & Simpan Model

## 1. Import Library

In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Image Processing
import cv2
from PIL import Image

# Sklearn — ML (UTS) & SVM
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    matthews_corrcoef
)
from sklearn.pipeline import Pipeline

# TensorFlow / Keras — VGG16
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU         : {tf.config.list_physical_devices('GPU')}")
print("Semua library berhasil diimpor!")

## 2. Konfigurasi

Struktur folder dataset yang dibutuhkan:
```
dataset/
├── melati_jakarta/     (360 gambar)
├── melati_jepang/      (360 gambar)
├── bintaro/            (360 gambar)
└── tapak_dara/         (360 gambar)
```

In [ ]:
# ============================================================
# KONFIGURASI PARAMETER
# ============================================================

# ⚠️ SESUAIKAN PATH DATASET ANDA
DATASET_PATH  = './dataset'

# Kelas
CLASS_NAMES   = ['melati_jakarta', 'melati_jepang', 'bintaro', 'tapak_dara']
CLASS_LABELS  = ['Melati Jakarta', 'Melati Jepang', 'Bintaro', 'Tapak Dara']
NUM_CLASSES   = len(CLASS_NAMES)
COLORS        = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

# Gambar
IMG_SIZE      = 224    # VGG16 standard input
IMG_SIZE_RF   = 64     # Untuk Random Forest

# Training
BATCH_SIZE    = 32
EPOCHS_P1     = 15     # Phase 1: head training
EPOCHS_P2     = 10     # Phase 2: fine-tuning
LR            = 0.001
TEST_SIZE     = 0.20
VAL_SIZE      = 0.10
RANDOM_STATE  = 42

# Random Forest
RF_ESTIMATORS = 100

# SVM kernel yang akan diuji
SVM_KERNELS   = ['linear', 'rbf', 'poly', 'sigmoid']

# Jumlah fitur dipilih dengan Chi-Square (25% dari total)
FEATURE_RATIO = 0.25

print("Konfigurasi berhasil disimpan.")
for k, v in {
    'Kelas': CLASS_NAMES, 'Input VGG16': f'{IMG_SIZE}x{IMG_SIZE}x3',
    'Batch size': BATCH_SIZE, 'Epochs P1/P2': f'{EPOCHS_P1}/{EPOCHS_P2}',
    'SVM kernels': SVM_KERNELS
}.items():
    print(f"  {k:<16}: {v}")

## 3. Load & Preprocessing Dataset

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset(path, class_names, img_size):
    X, y = [], []
    exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif')
    for label, name in enumerate(class_names):
        folder = os.path.join(path, name)
        if not os.path.exists(folder):
            print(f"⚠️  Folder tidak ada: {folder}")
            continue
        files = [f for f in os.listdir(folder) if f.lower().endswith(exts)]
        print(f"  [{name:<18}] {len(files):>4d} gambar")
        for f in files:
            try:
                img = Image.open(os.path.join(folder, f)).convert('RGB')
                img = img.resize((img_size, img_size))
                X.append(np.array(img, dtype=np.uint8))
                y.append(label)
            except Exception as e:
                print(f"    ⚠️  Skip {f}: {e}")
    return np.array(X), np.array(y)

print("Loading dataset...")
X_raw, y = load_dataset(DATASET_PATH, CLASS_NAMES, IMG_SIZE)

print(f"\nTotal  : {len(X_raw)} gambar | Shape: {X_raw.shape}")
for i, lbl in enumerate(CLASS_LABELS):
    print(f"  {lbl:<18}: {np.sum(y==i)} gambar")

In [ ]:
# ============================================================
# VISUALISASI SAMPEL DATASET
# ============================================================

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sampel Dataset Bunga', fontsize=15, fontweight='bold')
for i in range(NUM_CLASSES):
    idx_c = np.where(y == i)[0]
    samps = np.random.choice(idx_c, min(5, len(idx_c)), replace=False)
    for j, idx in enumerate(samps):
        axes[i, j].imshow(X_raw[idx])
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(CLASS_LABELS[i], fontsize=10,
                                  fontweight='bold', color=COLORS[i])
plt.tight_layout()
plt.savefig('sampel_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: sampel_dataset.png")

In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================

# VGG16: preprocess_input (zero-center per ImageNet channel mean)
X_vgg     = preprocess_input(X_raw.astype(np.float32))   # shape: (N,224,224,3)

# Random Forest: resize 64x64, flatten, normalisasi [0,1]
print("Mempersiapkan fitur pixel untuk Random Forest...")
X_rf_list = [cv2.resize(img, (IMG_SIZE_RF, IMG_SIZE_RF)) for img in X_raw]
X_rf_flat = np.array(X_rf_list, np.float32).reshape(len(X_rf_list), -1) / 255.0

# One-hot untuk VGG16 end-to-end
y_oh = to_categorical(y, NUM_CLASSES)

print(f"X_vgg shape    : {X_vgg.shape}   ← input VGG16")
print(f"X_rf_flat shape: {X_rf_flat.shape} ← input Random Forest")

In [ ]:
# ============================================================
# SPLIT TRAIN / VAL / TEST
# Mengacu: Fei et al. (2023) — 70% train, 30% test
# ============================================================

val_ratio = VAL_SIZE / (1 - TEST_SIZE)

# --- VGG16 ---
X_tv_v, X_test_v, y_tv_oh, y_test_oh, y_tv, y_test = train_test_split(
    X_vgg, y_oh, y,
    test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_train_v, X_val_v, y_train_oh, y_val_oh, y_train_v, y_val_v = train_test_split(
    X_tv_v, y_tv_oh, y_tv,
    test_size=val_ratio, stratify=y_tv, random_state=RANDOM_STATE
)

# --- Random Forest ---
X_tv_rf, X_test_rf, y_tv_rf, y_test_rf = train_test_split(
    X_rf_flat, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_train_rf, _, y_train_rf, _ = train_test_split(
    X_tv_rf, y_tv_rf, test_size=val_ratio, stratify=y_tv_rf, random_state=RANDOM_STATE
)

print("Pembagian Dataset:")
n = len(X_vgg)
for name, size in [('Train',len(X_train_v)),('Validasi',len(X_val_v)),('Test',len(X_test_v))]:
    print(f"  {name:<10}: {size:>4d} gambar ({size/n*100:.1f}%)")

## 4. Data Augmentation

Mengacu pada Fei et al. (2023) Section III-A-2: augmentasi meliputi horizontal flip, vertical flip, rotasi 90°, Gaussian noise, blur, dan perubahan kecerahan.

In [ ]:
# ============================================================
# DATA AUGMENTATION
# Referensi: Fei et al. (2023) Figure 5 — 8 teknik augmentasi
# ============================================================

train_aug = ImageDataGenerator(
    horizontal_flip   = True,           # (1) Horizontal flip
    vertical_flip     = False,          # (2) Vertical flip
    rotation_range    = 90,             # (4) Rotasi hingga 90°
    zoom_range        = 0.20,           # Scale
    width_shift_range = 0.15,
    height_shift_range= 0.15,
    brightness_range  = [0.7, 1.3],     # (8) Perubahan kecerahan
    shear_range       = 0.10,
    fill_mode         = 'nearest'
)
no_aug = ImageDataGenerator()

train_gen = train_aug.flow(X_train_v, y_train_oh, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = no_aug.flow(X_val_v, y_val_oh, batch_size=BATCH_SIZE, shuffle=False)

steps_ep  = max(1, len(X_train_v) // BATCH_SIZE)
val_steps = max(1, len(X_val_v) // BATCH_SIZE)

# ── Visualisasi 8 teknik augmentasi (sesuai Figure 5 jurnal) ──
sample = X_raw[0]
aug_vis = train_aug.flow(sample[np.newaxis].astype(np.float32), batch_size=1)

titles = ['Original', 'H-Flip', 'Rotasi', 'Zoom',
          'Shift', 'Brightness↑', 'Brightness↓', 'Shear', 'Combined']
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Teknik Data Augmentation (Fei et al., 2023 – Fig. 5)', fontsize=13, fontweight='bold')
axes[0,0].imshow(sample); axes[0,0].set_title('Original', fontweight='bold'); axes[0,0].axis('off')
for k in range(1, 10):
    aug_img = next(aug_vis)[0]
    aug_img = np.clip(aug_img + 123, 0, 255).astype(np.uint8)
    r, c = k // 5, k % 5
    axes[r, c].imshow(aug_img)
    axes[r, c].set_title(titles[k] if k < len(titles) else f'Aug {k}', fontsize=9)
    axes[r, c].axis('off')
plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: data_augmentation.png")

## 5. Metode ML (UTS) — Random Forest

In [ ]:
# ============================================================
# METODE ML (UTS): RANDOM FOREST
# ============================================================

print("=" * 58)
print("  METODE ML (UTS): RANDOM FOREST")
print("=" * 58)

rf_model = RandomForestClassifier(
    n_estimators=RF_ESTIMATORS, max_features='sqrt',
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)

def compute_metrics(y_true, y_pred, name):
    """Hitung dan tampilkan semua metrik evaluasi."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    # Specificity rata-rata
    cm   = confusion_matrix(y_true, y_pred)
    spec_list = []
    for i in range(NUM_CLASSES):
        tn = cm.sum() - (cm[i,:].sum() + cm[:,i].sum() - cm[i,i])
        fp = cm[:,i].sum() - cm[i,i]
        spec_list.append(tn/(tn+fp) if (tn+fp) > 0 else 0.0)
    spec = np.mean(spec_list)
    print(f"\n  {'─'*43}")
    print(f"  {name}")
    print(f"  {'─'*43}")
    print(f"  Accuracy     : {acc*100:.2f}%")
    print(f"  Precision    : {prec:.4f}")
    print(f"  Recall       : {rec:.4f}")
    print(f"  Specificity  : {spec:.4f}")
    print(f"  F1-Score     : {f1:.4f}")
    print(f"  MCC          : {mcc:.4f}")
    print(f"  {'─'*43}")
    return {'Model': name, 'Accuracy (%)': round(acc*100,2),
            'Precision': round(prec,4), 'Recall': round(rec,4),
            'Specificity': round(spec,4), 'F1-Score': round(f1,4),
            'MCC': round(mcc,4)}

rf_metrics = compute_metrics(y_test_rf, y_pred_rf, 'Random Forest (ML – UTS)')

print("\nClassification Report — Random Forest:")
print(classification_report(y_test_rf, y_pred_rf, target_names=CLASS_LABELS))

In [ ]:
# ─── Confusion Matrix Random Forest ───
cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS)
plt.title(f'Confusion Matrix — Random Forest\nAcc: {rf_metrics["Accuracy (%)"]:.2f}%',
          fontsize=12, fontweight='bold')
plt.ylabel('Aktual'); plt.xlabel('Prediksi')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('cm_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: cm_random_forest.png")

## 6. Metode Deep Learning (UAS) — VGG16 Fine-Tuning

Mengacu pada Fei et al. (2023): *"The common method is to use VGG, Inception, ResNet, and other classic architectures for transfer learning."*

In [ ]:
# ============================================================
# VGG16 END-TO-END FINE-TUNING
# ============================================================

print("=" * 58)
print("  VGG16 FINE-TUNING (End-to-End)")
print("=" * 58)

base_vgg = VGG16(weights='imagenet', include_top=False,
                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_vgg.trainable = False   # Freeze untuk Phase 1

x = base_vgg.output
x = GlobalAveragePooling2D(name='gap')(x)         # 512-D
x = Dense(1024, activation='relu', name='fc1')(x)
x = BatchNormalization(name='bn1')(x)
x = Dropout(0.5, name='drop1')(x)
x = Dense(256,  activation='relu', name='fc2')(x)
x = Dropout(0.3, name='drop2')(x)
out = Dense(NUM_CLASSES, activation='softmax', name='softmax')(x)

vgg_e2e = Model(inputs=base_vgg.input, outputs=out, name='VGG16_E2E')
vgg_e2e.compile(optimizer=Adam(LR), loss='categorical_crossentropy',
                metrics=['accuracy'])

cb_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_vgg16_p1.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

print(f"Phase 1: Training classification head (base frozen, max {EPOCHS_P1} epochs)...")
hist_p1 = vgg_e2e.fit(
    train_gen, steps_per_epoch=steps_ep, epochs=EPOCHS_P1,
    validation_data=val_gen, validation_steps=val_steps,
    callbacks=cb_p1, verbose=1
)

In [ ]:
# ─── Phase 2: Fine-tuning blok conv5 VGG16 ───
base_vgg.trainable = True
for layer in base_vgg.layers[:-4]:   # Freeze semua kecuali 4 layer terakhir
    layer.trainable = False

vgg_e2e.compile(optimizer=Adam(LR / 10), loss='categorical_crossentropy',
                metrics=['accuracy'])

cb_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_vgg16_finetuned.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

print(f"Phase 2: Fine-tuning blok conv5 VGG16 (max {EPOCHS_P2} epochs)...")
hist_p2 = vgg_e2e.fit(
    train_gen, steps_per_epoch=steps_ep, epochs=EPOCHS_P2,
    validation_data=val_gen, validation_steps=val_steps,
    callbacks=cb_p2, verbose=1
)
print("Training VGG16 selesai!")

In [ ]:
# ─── Grafik Training VGG16 ───
def merge_hist(h1, h2):
    return {k: h1.history[k] + h2.history.get(k, []) for k in h1.history}

hist_all = merge_hist(hist_p1, hist_p2)
ep_split = len(hist_p1.history['loss'])
epochs_r = range(1, len(hist_all['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History — VGG16 Fine-Tuning', fontsize=13, fontweight='bold')
for ax, (tk, vk), ylabel in zip(
        axes,
        [('accuracy','val_accuracy'), ('loss','val_loss')],
        ['Accuracy', 'Loss']):
    ax.plot(epochs_r, hist_all[tk], label='Training', color='#2196F3', lw=2)
    ax.plot(epochs_r, hist_all[vk], label='Validasi',
            color='#FF9800', ls='--', lw=2)
    ax.axvline(ep_split, color='red', ls=':', lw=1.5,
               alpha=0.7, label='Fine-tuning mulai')
    ax.set_title(f'Grafik {ylabel}'); ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafik_training_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: grafik_training_vgg16.png")

In [ ]:
# ─── Evaluasi VGG16 ───
y_prob_vgg = vgg_e2e.predict(X_test_v, batch_size=BATCH_SIZE, verbose=0)
y_pred_vgg = np.argmax(y_prob_vgg, axis=1)
vgg_metrics = compute_metrics(y_test, y_pred_vgg, 'VGG16 Fine-Tuning (DL – UAS)')
cm_vgg = confusion_matrix(y_test, y_pred_vgg)
print("\nClassification Report — VGG16:")
print(classification_report(y_test, y_pred_vgg, target_names=CLASS_LABELS))

## 7. Metode Hybrid (UAS) — VGG16 Feature Extractor + SVM

Mengacu pada Fei et al. (2023) Section I: *"CNN was proposed as a feature extractor and used in combination with other machine learning classification algorithms... such as VGG used as feature extractors and SVM used as feature classification tools."*

Tiga pendekatan hybrid:
- **CNN-SVM (semua fitur)** — fitur GAP langsung ke SVM
- **CNN-Chi²-SVM** — seleksi fitur Chi-Square → SVM (sesuai Koklu et al. 2022)
- **SVM kernel terbaik** — perbandingan Linear, RBF, Poly, Sigmoid

In [ ]:
# ============================================================
# EKSTRAKSI DEEP FEATURES dari VGG16 (Layer GAP)
# Referensi: Fei et al. (2023) — CNN sebagai feature extractor
# ============================================================

print("=" * 58)
print("  EKSTRAKSI DEEP FEATURES — VGG16 GAP Layer")
print("=" * 58)

# Model feature extractor: input → GlobalAveragePooling2D (512-D)
feat_extractor = Model(
    inputs  = vgg_e2e.input,
    outputs = vgg_e2e.get_layer('gap').output,
    name    = 'VGG16_FeatureExtractor'
)

# Gabungkan train+val+test untuk ekstraksi fitur
X_all_vgg = np.vstack([X_train_v, X_val_v, X_test_v])
y_all     = np.concatenate([y_train_v, y_val_v, y_test])

print(f"Mengekstrak fitur dari {len(X_all_vgg)} gambar...")
feats_all = feat_extractor.predict(X_all_vgg, batch_size=BATCH_SIZE, verbose=1)
print(f"Shape fitur diekstrak: {feats_all.shape}")

# Re-split ke train/test (proporsi sama)
n_tv  = len(X_train_v) + len(X_val_v)
F_train_all = feats_all[:n_tv];  y_F_train = y_all[:n_tv]
F_test      = feats_all[n_tv:];  y_F_test  = y_all[n_tv:]

print(f"Fitur train  : {F_train_all.shape}")
print(f"Fitur test   : {F_test.shape}")

In [ ]:
# ============================================================
# PERBANDINGAN 4 KERNEL SVM — TANPA SELEKSI FITUR
# ============================================================

print("=" * 58)
print("  VGG16 + SVM (Semua Fitur) — Perbandingan Kernel")
print("=" * 58)

scaler_svm = StandardScaler()
F_train_sc = scaler_svm.fit_transform(F_train_all)
F_test_sc  = scaler_svm.transform(F_test)

svm_all_results = []
svm_models      = {}

for kernel in SVM_KERNELS:
    print(f"\n  Training SVM kernel: {kernel.upper()}...")
    svm = SVC(kernel=kernel, C=1.0, gamma='scale',
              probability=True, random_state=RANDOM_STATE)
    svm.fit(F_train_sc, y_F_train)
    y_pred_svm = svm.predict(F_test_sc)

    acc  = accuracy_score(y_F_test, y_pred_svm)
    prec = precision_score(y_F_test, y_pred_svm, average='weighted', zero_division=0)
    rec  = recall_score(y_F_test, y_pred_svm, average='weighted', zero_division=0)
    f1   = f1_score(y_F_test, y_pred_svm, average='weighted', zero_division=0)
    mcc  = matthews_corrcoef(y_F_test, y_pred_svm)

    print(f"    Acc={acc*100:.2f}% | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")
    svm_all_results.append({
        'Model': f'VGG16 + SVM ({kernel.capitalize()}) Semua Fitur',
        'Accuracy (%)': round(acc*100, 2), 'Precision': round(prec,4),
        'Recall': round(rec,4), 'F1-Score': round(f1,4), 'MCC': round(mcc,4)
    })
    svm_models[f'svm_{kernel}_all'] = svm

# Model terbaik dari 4 kernel (tanpa seleksi fitur)
best_no_fs = max(svm_all_results, key=lambda x: x['Accuracy (%)'])
print(f"\n  ✅ Kernel terbaik (tanpa FS): {best_no_fs['Model']}")
print(f"     Accuracy: {best_no_fs['Accuracy (%)']}%")

In [ ]:
# ============================================================
# VGG16 + CHI-SQUARE FEATURE SELECTION + SVM
# Referensi: Koklu et al. (2022) + Fei et al. (2023)
# ============================================================

print("=" * 58)
print("  VGG16 + Chi-Square Seleksi Fitur + SVM")
print("=" * 58)

# Normalisasi MinMax (syarat Chi-Square: nilai non-negatif)
mm_scaler = MinMaxScaler()
F_train_mm = mm_scaler.fit_transform(F_train_all)
F_test_mm  = mm_scaler.transform(F_test)

n_feats_total    = F_train_mm.shape[1]
n_feats_selected = max(1, int(n_feats_total * FEATURE_RATIO))
print(f"  Total fitur    : {n_feats_total}")
print(f"  Fitur dipilih  : {n_feats_selected} ({FEATURE_RATIO*100:.0f}%)")

selector = SelectKBest(chi2, k=n_feats_selected)
F_train_sel = selector.fit_transform(F_train_mm, y_F_train)
F_test_sel  = selector.transform(F_test_mm)

# Scale setelah seleksi
scaler_fs = StandardScaler()
F_train_sel_sc = scaler_fs.fit_transform(F_train_sel)
F_test_sel_sc  = scaler_fs.transform(F_test_sel)

svm_fs_results = []
for kernel in SVM_KERNELS:
    print(f"\n  Training SVM+Chi² kernel: {kernel.upper()}...")
    svm_fs = SVC(kernel=kernel, C=1.0, gamma='scale',
                 probability=True, random_state=RANDOM_STATE)
    svm_fs.fit(F_train_sel_sc, y_F_train)
    y_pred_fs = svm_fs.predict(F_test_sel_sc)

    acc  = accuracy_score(y_F_test, y_pred_fs)
    prec = precision_score(y_F_test, y_pred_fs, average='weighted', zero_division=0)
    rec  = recall_score(y_F_test, y_pred_fs, average='weighted', zero_division=0)
    f1   = f1_score(y_F_test, y_pred_fs, average='weighted', zero_division=0)
    mcc  = matthews_corrcoef(y_F_test, y_pred_fs)

    print(f"    Acc={acc*100:.2f}% | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")
    svm_fs_results.append({
        'Model': f'VGG16 + Chi² + SVM ({kernel.capitalize()})',
        'Accuracy (%)': round(acc*100, 2), 'Precision': round(prec,4),
        'Recall': round(rec,4), 'F1-Score': round(f1,4), 'MCC': round(mcc,4)
    })
    svm_models[f'svm_{kernel}_fs'] = svm_fs

best_fs = max(svm_fs_results, key=lambda x: x['Accuracy (%)'])
print(f"\n  ✅ Kernel terbaik (+ Chi² FS): {best_fs['Model']}")
print(f"     Accuracy: {best_fs['Accuracy (%)']}%")

## 8. Perbandingan Semua Model

In [ ]:
# ============================================================
# TABEL PERBANDINGAN LENGKAP
# ============================================================

# Ambil hasil terbaik dari SVM per kelompok
best_svm_all = max(svm_all_results, key=lambda x: x['Accuracy (%)'])
best_svm_fs  = max(svm_fs_results,  key=lambda x: x['Accuracy (%)'])

# Tambahkan Specificity ke vgg_metrics
vgg_metrics.setdefault('Specificity', 0.0)
for m in [best_svm_all, best_svm_fs]:
    m.setdefault('Specificity', 0.0)

all_results = pd.DataFrame([
    rf_metrics,
    vgg_metrics,
    best_svm_all,
    best_svm_fs
])

print("\n" + "=" * 70)
print("  TABEL PERBANDINGAN SEMUA MODEL")
print("=" * 70)
cols = ['Model', 'Accuracy (%)', 'Precision', 'Recall', 'F1-Score', 'MCC']
print(all_results[cols].to_string(index=False))
print("=" * 70)

best_idx = all_results['Accuracy (%)'].idxmax()
print(f"\n  ✅ Model Terbaik : {all_results.loc[best_idx, 'Model']}")
print(f"     Accuracy       : {all_results.loc[best_idx, 'Accuracy (%)']}%")

delta = all_results.loc[best_idx, 'Accuracy (%)'] - rf_metrics['Accuracy (%)']
print(f"\n  📈 Peningkatan vs Random Forest: {delta:+.2f}%")

all_results[cols].to_csv('perbandingan_model.csv', index=False)
print("\n  Tabel disimpan: perbandingan_model.csv")

In [ ]:
# ============================================================
# TABEL DETAIL: SEMUA KERNEL SVM (Jurnal-style)
# ============================================================

df_svm_compare = pd.DataFrame(svm_all_results + svm_fs_results)
print("\nPerbandingan Detail Semua Kernel SVM:")
print(df_svm_compare[['Model','Accuracy (%)','Precision','Recall','F1-Score','MCC']].to_string(index=False))
df_svm_compare.to_csv('perbandingan_svm_kernels.csv', index=False)
print("\nDisimpan: perbandingan_svm_kernels.csv")

In [ ]:
# ============================================================
# VISUALISASI 1: BAR CHART AKURASI SEMUA MODEL
# ============================================================

model_labels = [
    'Random Forest\n(ML–UTS)',
    'VGG16\nFine-Tuning',
    f'VGG16+SVM\n({best_svm_all["Model"].split("(")[1].split(")")[0]})',
    f'VGG16+Chi²\n+SVM ({best_svm_fs["Model"].split("(")[1].split(")")[0]})'
]
accs   = all_results['Accuracy (%)'].values
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Perbandingan Performa: RF vs VGG16 vs VGG16+SVM',
             fontsize=13, fontweight='bold')

# Accuracy
bars = axes[0].bar(model_labels, accs, color=colors, edgecolor='black',
                   linewidth=0.8, width=0.5)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{acc:.2f}%', ha='center', va='bottom',
                 fontweight='bold', fontsize=10)
axes[0].set_ylim([max(0, min(accs)-15), 105])
axes[0].set_ylabel('Accuracy (%)', fontsize=11)
axes[0].set_title('Akurasi Model', fontsize=12)
axes[0].grid(True, axis='y', alpha=0.3)

# Multi-metric grouped bar
x    = np.arange(len(model_labels))
w    = 0.18
mcol = ['#FF5722', '#9C27B0', '#00BCD4', '#FF9800']
for i, (m, c) in enumerate(zip(['Precision','Recall','F1-Score','MCC'], mcol)):
    axes[1].bar(x + i*w, all_results[m].values, w, label=m, color=c,
                edgecolor='black', linewidth=0.4, alpha=0.87)
axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels(model_labels, fontsize=8)
axes[1].set_ylim([0, 1.15])
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Metrik Evaluasi Lengkap', fontsize=12)
axes[1].legend(fontsize=8); axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: perbandingan_model.png")

In [ ]:
# ============================================================
# VISUALISASI 2: GRAFIK KERNEL SVM (Jurnal-style)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perbandingan Kernel SVM: Tanpa vs Dengan Seleksi Fitur Chi²',
             fontsize=12, fontweight='bold')

kernels    = [k.capitalize() for k in SVM_KERNELS]
accs_all   = [r['Accuracy (%)'] for r in svm_all_results]
accs_fs    = [r['Accuracy (%)'] for r in svm_fs_results]
x_k = np.arange(len(kernels))
w   = 0.35

bars1 = axes[0].bar(x_k - w/2, accs_all, w, label='Semua Fitur',
                    color='#2196F3', edgecolor='black', linewidth=0.7)
bars2 = axes[0].bar(x_k + w/2, accs_fs, w, label=f'Chi² ({n_feats_selected} fitur)',
                    color='#FF9800', edgecolor='black', linewidth=0.7)
for bar in list(bars1)+list(bars2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{bar.get_height():.1f}', ha='center', fontsize=8, fontweight='bold')
axes[0].set_xticks(x_k); axes[0].set_xticklabels(kernels)
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_title('Akurasi per Kernel SVM')
axes[0].legend(); axes[0].grid(True, axis='y', alpha=0.3)
axes[0].set_ylim([max(0, min(accs_all+accs_fs)-10), 105])

# F1-Score
f1_all = [r['F1-Score'] for r in svm_all_results]
f1_fs  = [r['F1-Score'] for r in svm_fs_results]
axes[1].plot(kernels, f1_all, 'o-', label='Semua Fitur',
             color='#2196F3', lw=2, ms=8)
axes[1].plot(kernels, f1_fs,  's--', label=f'Chi² FS',
             color='#FF9800', lw=2, ms=8)
axes[1].set_ylabel('F1-Score'); axes[1].set_title('F1-Score per Kernel SVM')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig('perbandingan_svm_kernels.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: perbandingan_svm_kernels.png")

In [ ]:
# ============================================================
# VISUALISASI 3: CONFUSION MATRIX PANEL (4 Model Utama)
# ============================================================

# Dapatkan prediksi model SVM terbaik untuk confusion matrix
best_kernel_all = max(svm_all_results, key=lambda x: x['Accuracy (%)'])['Model']
best_kernel_all_key = best_kernel_all.split('(')[1].split(')')[0].lower()
best_kernel_fs  = max(svm_fs_results,  key=lambda x: x['Accuracy (%)'])['Model']
best_kernel_fs_key  = best_kernel_fs.split('(')[1].split(')')[0].lower()

y_pred_svm_best    = svm_models[f'svm_{best_kernel_all_key}_all'].predict(F_test_sc)
y_pred_svm_fs_best = svm_models[f'svm_{best_kernel_fs_key}_fs'].predict(F_test_sel_sc)

cms   = [cm_rf, cm_vgg,
         confusion_matrix(y_F_test, y_pred_svm_best),
         confusion_matrix(y_F_test, y_pred_svm_fs_best)]
titles = [
    f'Random Forest (ML–UTS)\n{rf_metrics["Accuracy (%)"]:.2f}%',
    f'VGG16 Fine-Tuning\n{vgg_metrics["Accuracy (%)"]:.2f}%',
    f'VGG16+SVM ({best_kernel_all_key.capitalize()})\n{best_svm_all["Accuracy (%)"]:.2f}%',
    f'VGG16+Chi²+SVM ({best_kernel_fs_key.capitalize()})\n{best_svm_fs["Accuracy (%)"]:.2f}%'
]
cmaps  = ['Blues', 'Greens', 'Oranges', 'Purples']

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrix — Semua Model Utama', fontsize=14, fontweight='bold')
for ax, cm_i, title, cmap in zip(axes.flat, cms, titles, cmaps):
    sns.heatmap(cm_i, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Aktual'); ax.set_xlabel('Prediksi')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('confusion_matrix_panel.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: confusion_matrix_panel.png")

In [ ]:
# ============================================================
# VISUALISASI 4: METRIK PER KELAS
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

prf_rf  = precision_recall_fscore_support(y_test_rf, y_pred_rf, average=None, zero_division=0)
prf_vgg = precision_recall_fscore_support(y_test, y_pred_vgg, average=None, zero_division=0)
prf_svm = precision_recall_fscore_support(y_F_test, y_pred_svm_fs_best, average=None, zero_division=0)

x     = np.arange(NUM_CLASSES)
width = 0.26
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Perbandingan Metrik Per Kelas', fontsize=13, fontweight='bold')

for ax, rf_v, vgg_v, svm_v, metric in zip(
        axes,
        [prf_rf[0], prf_rf[1], prf_rf[2]],
        [prf_vgg[0], prf_vgg[1], prf_vgg[2]],
        [prf_svm[0], prf_svm[1], prf_svm[2]],
        ['Precision', 'Recall', 'F1-Score']):
    ax.bar(x - width,   rf_v,  width, label='Random Forest', color='#2196F3',
           edgecolor='black', lw=0.5, alpha=0.9)
    ax.bar(x,           vgg_v, width, label='VGG16', color='#4CAF50',
           edgecolor='black', lw=0.5, alpha=0.9)
    ax.bar(x + width,   svm_v, width,
           label=f'VGG16+Chi²+SVM ({best_kernel_fs_key.capitalize()})',
           color='#9C27B0', edgecolor='black', lw=0.5, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_LABELS, rotation=20, ha='right', fontsize=8)
    ax.set_ylim([0, 1.15]); ax.set_title(metric); ax.set_ylabel('Score')
    ax.legend(fontsize=7); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrik_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: metrik_per_kelas.png")

## 9. Demo Prediksi Input Baru

In [ ]:
# ============================================================
# DEMO PREDIKSI DARI TEST SET (8 sampel)
# ============================================================

n_demo   = 8
demo_idx = np.random.choice(len(X_test_v), n_demo, replace=False)
X_demo   = X_test_v[demo_idx]

# Prediksi VGG16
prob_vgg_d = vgg_e2e.predict(X_demo, verbose=0)
pred_vgg_d = np.argmax(prob_vgg_d, axis=1)

# Prediksi VGG16+SVM (model terbaik)
feat_demo = feat_extractor.predict(X_demo, verbose=0)
feat_demo_mm = mm_scaler.transform(feat_demo)
feat_demo_sel = selector.transform(feat_demo_mm)
feat_demo_sc  = scaler_fs.transform(feat_demo_sel)
pred_svm_d = svm_models[f'svm_{best_kernel_fs_key}_fs'].predict(feat_demo_sc)

true_demo  = y_test[demo_idx]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Demo Prediksi — VGG16 Fine-Tuning vs VGG16+SVM', fontsize=13, fontweight='bold')

for k, ax in enumerate(axes.flat):
    # Tampilkan gambar asli (denormalisasi kasar)
    img_show = np.clip(X_demo[k] + 123, 0, 255).astype(np.uint8)
    ax.imshow(img_show)
    true_lbl = CLASS_LABELS[true_demo[k]]
    vgg_lbl  = CLASS_LABELS[pred_vgg_d[k]]
    svm_lbl  = CLASS_LABELS[pred_svm_d[k]]
    conf_vgg = prob_vgg_d[k][pred_vgg_d[k]] * 100

    ok_vgg = '✓' if pred_vgg_d[k] == true_demo[k] else '✗'
    ok_svm = '✓' if pred_svm_d[k] == true_demo[k] else '✗'
    color  = 'green' if ok_vgg == '✓' and ok_svm == '✓' else \
              ('orange' if ok_vgg == '✓' or ok_svm == '✓' else 'red')

    ax.set_title(
        f'Aktual: {true_lbl}\n'
        f'VGG16: {vgg_lbl} {ok_vgg} ({conf_vgg:.0f}%)\n'
        f'SVM: {svm_lbl} {ok_svm}',
        fontsize=7.5, color=color, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('demo_prediksi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: demo_prediksi.png")

In [ ]:
# ============================================================
# FUNGSI PREDIKSI GAMBAR BARU (dari file)
# ============================================================

def predict_new_image(image_path):
    """
    Prediksi kelas bunga dari file gambar baru menggunakan
    dua model: VGG16 Fine-Tuning & VGG16+SVM.

    Args:
        image_path (str): path ke file gambar
    """
    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32)
    arr_pre  = preprocess_input(arr)
    arr_batch = arr_pre[np.newaxis]

    # VGG16 end-to-end
    prob_vgg = vgg_e2e.predict(arr_batch, verbose=0)[0]
    pred_vgg = np.argmax(prob_vgg)

    # VGG16 + SVM
    feat = feat_extractor.predict(arr_batch, verbose=0)
    feat_mm  = mm_scaler.transform(feat)
    feat_sel = selector.transform(feat_mm)
    feat_sc  = scaler_fs.transform(feat_sel)
    pred_svm = svm_models[f'svm_{best_kernel_fs_key}_fs'].predict(feat_sc)[0]
    prob_svm = svm_models[f'svm_{best_kernel_fs_key}_fs'].predict_proba(feat_sc)[0]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(np.array(img)); axes[0].axis('off')
    axes[0].set_title('Input Gambar', fontweight='bold')

    for ax, probs, pred, title in zip(
            axes[1:],
            [prob_vgg, prob_svm],
            [pred_vgg, pred_svm],
            ['VGG16 Fine-Tuning', f'VGG16+Chi²+SVM ({best_kernel_fs_key.capitalize()})']):
        bar_c = ['#4CAF50' if i==pred else '#B0BEC5' for i in range(NUM_CLASSES)]
        ax.barh(CLASS_LABELS, probs*100, color=bar_c, edgecolor='black', lw=0.5)
        ax.set_xlim([0, 110]); ax.set_xlabel('Probabilitas (%)')
        ax.set_title(f'{title}\n→ {CLASS_LABELS[pred]} ({probs[pred]*100:.1f}%)',
                     fontweight='bold', color='#2E7D32')
        for i, v in enumerate(probs*100):
            ax.text(v+1, i, f'{v:.1f}%', va='center', fontsize=8)
        ax.grid(True, axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('prediksi_gambar_baru.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"VGG16 : {CLASS_LABELS[pred_vgg]} ({prob_vgg[pred_vgg]*100:.2f}%)")
    print(f"CNN-SVM: {CLASS_LABELS[pred_svm]} ({prob_svm[pred_svm]*100:.2f}%)")

print("Fungsi predict_new_image() siap.")
print("Contoh: predict_new_image('foto_bunga.jpg')")

## 10. Simpan Model & Ringkasan

In [ ]:
# ============================================================
# SIMPAN SEMUA MODEL
# ============================================================

vgg_e2e.save('model_vgg16_e2e.h5')
feat_extractor.save('model_vgg16_feat_extractor.h5')
joblib.dump(rf_model,     'model_random_forest.pkl')
joblib.dump(svm_models,   'model_svm_all_kernels.pkl')
joblib.dump(mm_scaler,    'scaler_minmax.pkl')
joblib.dump(selector,     'chi2_selector.pkl')
joblib.dump(scaler_fs,    'scaler_std_fs.pkl')
joblib.dump(scaler_svm,   'scaler_std_all.pkl')
with open('class_names.json', 'w') as f:
    json.dump({'class_names': CLASS_NAMES, 'class_labels': CLASS_LABELS,
               'best_kernel': best_kernel_fs_key}, f)

print("✅ Semua model berhasil disimpan:")
files = [
    'model_vgg16_e2e.h5', 'model_vgg16_feat_extractor.h5',
    'model_random_forest.pkl', 'model_svm_all_kernels.pkl',
    'scaler_minmax.pkl', 'chi2_selector.pkl', 'scaler_std_fs.pkl',
    'class_names.json'
]
for f in files: print(f"   • {f}")

In [ ]:
# ============================================================
# RINGKASAN AKHIR
# ============================================================

print("\n" + "=" * 68)
print("  RINGKASAN HASIL PENELITIAN")
print("  Klasifikasi Bunga: Melati Jakarta | Melati Jepang | Bintaro | Tapak Dara")
print("=" * 68)
print(f"  Dataset      : {len(X_raw)} gambar | {NUM_CLASSES} kelas | ~360 per kelas")
print(f"  Split        : Train {int((1-TEST_SIZE-VAL_SIZE)*100)}% | Val {int(VAL_SIZE*100)}% | Test {int(TEST_SIZE*100)}%")
print(f"  Augmentasi   : H-Flip, Rotasi 90°, Zoom, Shift, Brightness, Shear")
print(f"  Deep Feature : VGG16 GAP Layer → {feats_all.shape[1]} fitur")
print(f"  Setelah Chi² : {n_feats_selected} fitur ({FEATURE_RATIO*100:.0f}%)")
print(f"  Kernel SVM   : {', '.join(SVM_KERNELS)}")
print(f"  Referensi    : Fei et al. (2023) — IEEE Access 11, 17283")
print()
print(f"  {'Model':<45} {'Acc (%)':>8} {'F1':>8} {'MCC':>8}")
print("  " + "-" * 69)
for _, row in all_results.iterrows():
    marker = ' ←BEST' if row['Accuracy (%)'] == all_results['Accuracy (%)'].max() else ''
    print(f"  {row['Model']:<45} {row['Accuracy (%)']:>7.2f}% {row['F1-Score']:>8.4f} {row['MCC']:>8.4f}{marker}")
print("  " + "-" * 69)

delta = all_results['Accuracy (%)'].max() - rf_metrics['Accuracy (%)']
print(f"\n  📈 Peningkatan DL+SVM vs ML (RF) : {delta:+.2f}%")
print()
print("  📁 Output Files:")
outs = [
    'sampel_dataset.png', 'data_augmentation.png',
    'cm_random_forest.png', 'grafik_training_vgg16.png',
    'confusion_matrix_panel.png', 'perbandingan_model.png',
    'perbandingan_svm_kernels.png', 'metrik_per_kelas.png',
    'demo_prediksi.png', 'perbandingan_model.csv',
    'perbandingan_svm_kernels.csv', 'model_vgg16_e2e.h5',
    'model_svm_all_kernels.pkl'
]
for f in outs: print(f"     • {f}")
print("=" * 68)